# 01 · Exploratory data analysis of the raw sources

Exploration only - no production logic lives here. Findings from this notebook drive the silver cleaning rules in `utils/transformation/build_silver.py` and the join design in `utils/features/feature_store.py`.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.width', 200)

lms = pd.read_csv('../data/lms_loan_daily.csv')
attributes = pd.read_csv('../data/features_attributes.csv', dtype=str)
financials = pd.read_csv('../data/features_financials.csv', dtype=str)
clickstream = pd.read_csv('../data/feature_clickstream.csv')

for name, df in [('lms_loan_daily', lms), ('attributes', attributes), ('financials', financials), ('clickstream', clickstream)]:
    print(f"{name:15s} rows={len(df):>7,}  customers={df.Customer_ID.nunique():>6,}  snapshots {df.snapshot_date.min()} .. {df.snapshot_date.max()}")

lms_loan_daily  rows=137,500  customers=12,500  snapshots 2023-01-01 .. 2025-11-01
attributes      rows= 12,500  customers=12,500  snapshots 2023-01-01 .. 2025-01-01
financials      rows= 12,500  customers=12,500  snapshots 2023-01-01 .. 2025-01-01
clickstream     rows=215,376  customers= 8,974  snapshots 2023-01-01 .. 2024-12-01


## 1. Grain and join key
Attributes and financials hold **one row per customer**, and that row's `snapshot_date` equals the customer's `loan_start_date`. The application date is therefore a leakage-free join key between features and labels.

In [2]:
loan_starts = lms[lms.installment_num == 0][['Customer_ID', 'loan_start_date']]
m = loan_starts.merge(attributes[['Customer_ID', 'snapshot_date']], on='Customer_ID')
print('attribute snapshot == loan start date:', (m.snapshot_date == m.loan_start_date).mean())

cs = clickstream.groupby('Customer_ID').snapshot_date.min().rename('first_click')
m = loan_starts.merge(clickstream[['Customer_ID', 'snapshot_date']], left_on=['Customer_ID', 'loan_start_date'], right_on=['Customer_ID', 'snapshot_date'], how='left')
print('loans with clickstream in the application month:', round(m.snapshot_date.notna().mean(), 3))

attribute snapshot == loan start date: 1.0


loans with clickstream in the application month: 0.718


## 2. Dirty values in attributes and financials

In [3]:
def to_num(s):
    return pd.to_numeric(s.str.replace('_', '', regex=False).str.strip(), errors='coerce')

print('ages with a stray "_":', attributes.Age.str.contains('_').sum())
age = to_num(attributes.Age)
print('ages outside 14-100:', (~age.between(14, 100)).sum(), '| min', age.min(), 'max', age.max())
print('placeholder occupations:', (attributes.Occupation == '_______').sum())
print('malformed SSNs:', (~attributes.SSN.str.match(r'^\d{3}-\d{2}-\d{4}$')).sum())

ages with a stray "_": 637
ages outside 14-100: 319 | min -500 max 8678
placeholder occupations: 880
malformed SSNs: 703


In [4]:
rows = []
for col in financials.columns.drop(['Customer_ID', 'snapshot_date', 'Type_of_Loan', 'Credit_History_Age']):
    s = financials[col].astype(str)
    non_numeric = s[~s.str.match(r'^-?\d+(\.\d+)?$')]
    rows.append((col, len(non_numeric), non_numeric.value_counts().head(3).to_dict()))
pd.DataFrame(rows, columns=['column', 'non-numeric values', 'most common']).query('`non-numeric values` > 0')

,column,non-numeric values,most common
0,Annual_Income,859,"{'40380.18_': 1, '52312.68_': 1, '49718.55_': 1}"
5,Num_of_Loan,623,"{'2_': 110, '4_': 105, '3_': 79}"
7,Num_of_Delayed_Payment,374,"{'18_': 25, '19_': 24, '20_': 23}"
8,Changed_Credit_Limit,254,{'_': 254}
10,Credit_Mix,12500,"{'Standard': 4497, 'Good': 3032, '_': 2611}"
11,Outstanding_Debt,139,"{'2699.17_': 1, '642.42_': 1, '755.17_': 1}"
13,Payment_of_Min_Amount,12500,"{'Yes': 6571, 'No': 4491, 'NM': 1438}"
15,Amount_invested_monthly,558,{'__10000__': 558}
16,Payment_Behaviour,12500,"{'Low_spent_Small_value_payments': 3202, 'High..."
17,Monthly_Balance,1,{'__-333333333333333333333333333__': 1}


In [5]:
numeric = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Num_of_Delayed_Payment', 'Num_Credit_Inquiries', 'Annual_Income', 'Total_EMI_per_month']
financials[numeric].apply(to_num).quantile([0, .01, .5, .95, .99, 1]).T.round(1)

,0.00,0.01,0.50,0.95,0.99,1.00
Num_Bank_Accounts,-1.0,0.0,6.0,10.0,511.0,1756.0
Num_Credit_Card,0.0,1.0,5.0,10.0,878.0,1499.0
Interest_Rate,1.0,1.0,14.0,33.0,2865.0,5789.0
Num_of_Loan,-100.0,-100.0,3.0,8.0,9.0,1495.0
Num_of_Delayed_Payment,-3.0,0.0,14.0,24.0,27.0,4293.0
Num_Credit_Inquiries,0.0,0.0,6.0,14.0,946.5,2554.0
Annual_Income,7005.9,7542.4,37572.4,134316.5,178815.7,23834698.0
Total_EMI_per_month,0.0,0.0,72.9,592.3,57478.9,81971.0


The 99th-100th percentiles show corrupted outliers (e.g. 1,756 bank accounts, 5,789% interest). Silver nulls values outside a fixed plausible range rather than clipping to a percentile - a percentile computed on all data would leak test information.

In [6]:
ratio = to_num(financials.Annual_Income) / (12 * to_num(financials.Monthly_Inhand_Salary))
print('annual income / (12 x monthly salary) quantiles:', ratio.quantile([.5, .95, .99]).round(2).to_dict())
print('incomes > 3x annual salary:', (ratio > 3).sum())
print('EMI > monthly salary:', (to_num(financials.Total_EMI_per_month) > to_num(financials.Monthly_Inhand_Salary)).sum())

annual income / (12 x monthly salary) quantiles: {0.5: 1.0, 0.95: 1.22, 0.99: 1.7}
incomes > 3x annual salary: 114
EMI > monthly salary: 397


In [7]:
types = financials.Type_of_Loan.fillna('')
listed = types.apply(lambda s: 0 if s == '' else len(s.split(',')))
print('Num_of_Loan equals number of listed loan types (valid rows):')
pd.crosstab(to_num(financials.Num_of_Loan).clip(-1, 10), listed)

Num_of_Loan equals number of listed loan types (valid rows):


Type_of_Loan,0,1,2,3,4,5,6,7,8,9
Num_of_Loan,,,,,,,,,,
-1,53,60,75,72,77,42,37,40,17,27
0,1363,0,0,0,0,0,0,0,0,0
1,0,1324,0,0,0,0,0,0,0,0
2,0,0,1879,0,0,0,0,0,0,0
3,0,0,0,1887,0,0,0,0,0,0
4,0,0,0,0,1844,0,0,0,0,0
5,0,0,0,0,0,894,0,0,0,0
6,0,0,0,0,0,0,976,0,0,0
7,0,0,0,0,0,0,0,916,0,0


## 3. Label balance

In [8]:
mob6 = lms[lms.installment_num == 6].copy()
mob6['installments_missed'] = np.ceil(mob6.overdue_amt / mob6.due_amt.replace(0, np.nan)).fillna(0)
print('share of loans with any overdue amount at MOB 6:', round((mob6.overdue_amt > 0).mean(), 3))

share of loans with any overdue amount at MOB 6: 0.288
